# nn-module-subclass — ex1: minimal stateless Module (no __init__)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-module-subclass`. Running the final beacon cell reports progress against the `PyTorch: nn.Module subclassing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Module subclassing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-module-subclass`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-module-subclass"
DD_SUBTOPIC = "PyTorch: nn.Module subclassing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Module` subclassing — quick refresher

Every learnable building block in PyTorch is an `nn.Module` subclass. The minimal pattern is:

```python
class MyLayer(nn.Module):
    def __init__(self, ...):
        super().__init__()      # MUST be first — wires up _parameters / _modules dicts
        self.weight = nn.Parameter(...)
    def forward(self, x):
        return ...              # never call .forward() directly — use module(x)
```

**Two non-obvious rules.**
1. If `__init__` does anything (assigns Parameters, sub-Modules, or buffers), it MUST call `super().__init__()` first. Forgetting this raises `AttributeError: cannot assign parameter before Module.__init__() call`.
2. Modules with no state can omit `__init__` entirely and just define `forward` (e.g. ARENA's `ReLU`). The base `nn.Module.__init__` runs implicitly.

**Call convention.** Use `module(x)`, never `module.forward(x)` — the `__call__` wrapper runs hooks (pre/post forward, gradient hooks) that you lose by calling forward directly.

### Exercise 1 — minimal stateless Module (no __init__)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Define an nn.Module subclass that has no state — only a forward method — and invoke it via __call__.
> Keywords: nn.Module, forward, stateless, ReLU-like
> ```

**KCs targeted:** `module-subclass-trivial-forward`, `module-call-via-dunder-call`

Implement `ex1_make_square_module()`. The minimal possible Module:

1. Define a class `SquareLayer` that subclasses `t.nn.Module`.
2. Do NOT write an `__init__` method — the layer has no state.
3. Implement `forward(self, x)` returning `x ** 2` (elementwise).
4. Return an INSTANCE of `SquareLayer` from `ex1_make_square_module()`.

This mirrors ARENA's first nn.Module exercise (the trivial ReLU): when there's no state to declare, you can skip `__init__` entirely and the base `nn.Module.__init__` runs implicitly.

The test calls the module via `module(x)` (NOT `module.forward(x)`) to confirm the `__call__` plumbing works without you wiring anything up.

In [ ]:
def ex1_make_square_module():
    """Return an instance of a stateless nn.Module that squares its input."""
    raise NotImplementedError()


def _test_ex1():
    module = ex1_make_square_module()
    # Must be an nn.Module subclass instance.
    assert isinstance(module, t.nn.Module), f'expected nn.Module, got {type(module).__name__}'
    # Must NOT define __init__ (i.e. inherits the base Module.__init__).
    assert '__init__' not in type(module).__dict__, (
        'this exercise tests the no-__init__ pattern; remove your __init__ override'
    )
    # forward must square elementwise.
    x = t.tensor([-2.0, -1.0, 0.0, 1.0, 3.0])
    y = module(x)
    expected = t.tensor([4.0, 1.0, 0.0, 1.0, 9.0])
    assert t.allclose(y, expected), f'expected {expected}, got {y}'
    # No state — parameters() must be empty.
    assert list(module.parameters()) == [], 'a stateless module should have zero parameters'
    # state_dict must be empty too.
    assert len(module.state_dict()) == 0, 'a stateless module should have an empty state_dict'
    # Calling via __call__ vs .forward must give the same result
    # (this is the only difference that matters for hooks downstream).
    x2 = t.randn(3, 4)
    assert t.allclose(module(x2), module.forward(x2)), 'module(x) and module.forward(x) must agree on output'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_make_square_module():
    class SquareLayer(t.nn.Module):
        def forward(self, x):
            return x ** 2
    return SquareLayer()
```

**Why no `__init__` works.** `nn.Module.__init__` is what creates `self._parameters`, `self._modules`, `self._buffers` (the dicts auto-registration writes to). When you don't override it, Python calls the base version automatically when you instantiate the class. No state → no need to register anything → no `__init__` needed.

**Why use `module(x)` not `module.forward(x)`.** The base class implements `__call__` which runs registered pre/post-forward hooks and then calls your `forward`. Calling `.forward` directly skips the hooks — fine for this trivial case, fatal once you attach gradient checkpointing, profiling, or grad hooks.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()